In [5]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

VECTORSTORE_PATH = r"C:\Users\David\Documents\AeroGPT\data\vectorStores\regulatory_store"

embeddings = OpenAIEmbeddings()

store = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

query = "What are the FAA requirements for flight crew training?"
results = store.similarity_search(query, k=4)

for r in results:
    print("---")
    print("Contenido:", r.page_content[:400])
    print("Fuente:", r.metadata.get("source"))


---
Contenido: s. Working
groups in three major areas were formed: (1) man/machine interface; (2) flightcrew member
training; and (3) operating environment. Each working group submitted a report and
recommendations to the joint task force. On June 8, 1988, the recommendations of the joint task
force were presented to Administrator McArtor. The major substantive recommendations to the
Administrator from the fligh
Fuente: AC_120-54A_CHG_1.txt
---
Contenido: er required ground training subjects may be found in various FAA publications.
1.6.3 Flight Training and Checking. Flight training is instruction, practice, and review that
provides individuals with the practical, hands-on experience of integrating knowledge
and cognitive skills with the psychomotor skills necessary to perform the tasks required
for the duty position. Flight checking is a practica
Fuente: AC_120-114.txt
---
Contenido: .7.1 Regulatory Requirements. In accordance with § 121.401, each air carrier must provide
and keep cu

In [9]:
# ========================================================
#                   AEROGPT — RAG EN ESPAÑOL
# ========================================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS


class AeroGPT:
    def __init__(
        self,
        regulatory_store_path,
        technical_store_path,
        model_creative="gpt-4.1",
        temperature_creative=0.7,
        max_tokens_creative=600,
    ):
        # -------------------------------
        # 1) MODELOS
        # -------------------------------
        self.llm_creative = ChatOpenAI(
            model=model_creative,
            temperature=temperature_creative,
            max_tokens=max_tokens_creative
        )

        self.llm_system = ChatOpenAI(
            model="gpt-4.1-mini",
            temperature=0.0,
            max_tokens=300
        )

        self.embeddings = OpenAIEmbeddings()

        # -------------------------------
        # 2) VECTORSTORES
        # -------------------------------
        self.regulatory_store = FAISS.load_local(
            regulatory_store_path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )

        self.technical_store = FAISS.load_local(
            technical_store_path,
            self.embeddings,
            allow_dangerous_deserialization=True
        )

    # --------------------------------------------------
    #               TRADUCTOR AL INGLÉS
    # --------------------------------------------------
    def translate_to_english(self, text):
        prompt = f"Translate the following text to English. Output ONLY the translation:\n{text}"
        return self.llm_system.invoke(prompt).content

    # --------------------------------------------------
    #               RAG BASE — EN ESPAÑOL
    # --------------------------------------------------
    def rag_search(self, query, store):
        # 1) Traducir a inglés para mejor búsqueda
        query_en = self.translate_to_english(query)

        # 2) Recuperar chunks
        docs = store.similarity_search(query_en, k=4)
        context = "\n\n".join(d.page_content for d in docs)

        # 3) Prompt para el LLM creativo
        rag_prompt = f"""
Eres un asistente experto en aviación.
Responde SIEMPRE en español, incluso si el contexto está en inglés.

Usa SOLO el siguiente contexto:

{context}

Pregunta del usuario:
{query}

Responde de forma clara, técnica y precisa en español:
"""
        return self.llm_creative.invoke(rag_prompt).content

    # --------------------------------------------------
    #             MÉTODOS ESPECIALIZADOS
    # --------------------------------------------------
    def ask_regulatory(self, question):
        """Consultas FAA + EASA"""
        return self.rag_search(question, self.regulatory_store)

    def ask_technical(self, question):
        """Consultas Airbus FAST / técnico"""
        return self.rag_search(question, self.technical_store)

    # --------------------------------------------------
    #             AUTOSELECTOR GENERAL
    # --------------------------------------------------
    def ask(self, question):
        """Detecta si es normativa o técnico y enruta automáticamente"""

        routing_prompt = f"""
Clasifica esta pregunta en una de estas categorías:
1. "regulatorio" → normas FAA, EASA, reglas, AC, CS-XX, certificación, requisitos
2. "tecnico" → Airbus FAST, ingeniería, aerodinámica, mantenimiento técnico

Pregunta: {question}

Responde SOLO con: regulatorio o tecnico
"""

        cat = self.llm_system.invoke(routing_prompt).content.strip().lower()

        if "reg" in cat:
            return self.ask_regulatory(question)
        else:
            return self.ask_technical(question)


In [10]:
aero = AeroGPT(
    regulatory_store_path=r"C:\Users\David\Documents\AeroGPT\data\vectorStores\regulatory_store",
    technical_store_path=r"C:\Users\David\Documents\AeroGPT\data\vectorStores\technical_store",
    model_creative="gpt-4.1",
    temperature_creative=0.7
)

respuesta = aero.ask("¿Cuáles son los requisitos de la FAA para la certificación de aeronavegabilidad?")
print(respuesta)


Los requisitos de la FAA para la certificación de aeronavegabilidad son los siguientes:

1. **Conformidad con el Certificado de Tipo**: El producto (aeronave, motor o hélice) debe conformarse a su Certificado de Tipo (TC), lo que significa que la configuración y los componentes instalados deben coincidir con los datos, dibujos y especificaciones que forman parte del TC, incluyendo cualquier Certificado de Tipo Suplementario (STC), Directivas de Aeronavegabilidad (AD) y alteraciones aprobadas en campo.

2. **Condición para operación segura**: El producto debe estar en condiciones para operar de manera segura.

3. **Efectividad del Certificado de Aeronavegabilidad**: El certificado estándar de aeronavegabilidad de EE. UU. es válido mientras no sea entregado, suspendido, revocado o tenga una fecha de terminación establecida por el Administrador, y solamente mientras:
   - Se realice el mantenimiento, mantenimiento preventivo y alteraciones conforme a las partes 43 y 91 de 14 CFR.
   - La 

In [11]:
print(aero.ask_regulatory("Explica los límites de fatiga en EASA CS-25"))
print(aero.ask_technical("¿Cómo afecta el ángulo de ataque a la sustentación según Airbus FAST?"))
print(aero.ask("¿Qué documentación requiere FAA para mantenimiento en línea?"))


En EASA CS-25, los límites de fatiga están relacionados con la evaluación y gestión de la tolerancia al daño y la fatiga estructural de aeronaves de transporte. Aunque tu contexto hace referencia principalmente a la normativa y evolución de la FAA (como FAR/CS 25.571 y los conceptos de LOV), EASA ha armonizado sus requisitos con los de la FAA en este aspecto.

**Límites de fatiga en EASA CS-25:**

1. **Evaluación de estructura frente a fatiga y tolerancia al daño:**  
Las aeronaves certificadas bajo CS-25 deben demostrar que su estructura es capaz de resistir la fatiga y el daño por grietas a lo largo de su vida operativa prevista, bajo las condiciones y cargas esperadas.

2. **Enfoque “Fail-safe” y “Fatigue strength”:**  
Se reconoce que, en algunos casos, no es posible evitar completamente la aparición de grietas por fatiga. Por ello, la normativa exige que las aeronaves sean diseñadas bajo el enfoque de “fatigue strength” (fuerza a fatiga, donde se busca que no ocurran grietas) o “f

In [12]:
print(aero.ask("¿Como se hace una paella?"))


Lo siento, pero tu pregunta sobre cómo se hace una paella no está relacionada con la aviación ni se encuentra dentro del contexto proporcionado. Si tienes alguna consulta sobre galley, catering, o sistemas de cabina en aviación, estaré encantado de ayudarte.
